# 📉 Chương 4: Phân Tích Tự Tương Quan (ACF & PACF)
## Advanced Data Science - Session 5

---

**Mục tiêu chương này:**
- Hiểu Autocorrelation (ACF) và Partial Autocorrelation (PACF)
- Biết cách đọc và diễn giải ACF/PACF plots
- Sử dụng ACF/PACF để xác định tham số cho ARIMA model
- Thực hành nhận biết AR, MA, ARMA từ ACF/PACF

## 4.1 Autocorrelation (ACF) Là Gì?

### 🎯 Ví dụ minh họa:

Tưởng tượng bạn đo **nhiệt độ Hà Nội mỗi ngày**:
- Nếu hôm nay **35°C**, thì ngày mai có thể cũng **34-36°C** → Tương quan **cao** (lag 1)
- Nếu hôm nay **35°C**, thì 1 tuần sau có thể **30-37°C** → Tương quan **vừa** (lag 7)
- Nếu hôm nay **35°C**, thì 6 tháng sau có thể **15°C** (mùa đông) → Tương quan **thấp/âm** (lag 180)

→ **ACF** đo lường mức độ tương quan này ở mỗi **lag**.

### 📐 Công thức:

```
ACF(k) = Corr(Y_t, Y_{t-k}) = Cov(Y_t, Y_{t-k}) / Var(Y_t)
```

- **k = 0:** ACF = 1 (tương quan với chính nó)
- **k = 1:** Tương quan với quan sát trước 1 bước
- **k = 2:** Tương quan với quan sát trước 2 bước

### 🔑 Đặc điểm quan trọng:
- ACF bao gồm cả **trực tiếp** và **gián tiếp** correlation
- Ví dụ: ACF(2) bao gồm cả ảnh hưởng qua lag 1 → lag 2

## 4.2 Partial Autocorrelation (PACF) Là Gì?

### 🎯 Ví dụ minh họa:

Trong một gia đình 3 thế hệ:
- Ông → Bố → Con
- **ACF:** Con giống Ông? → Có (nhưng qua Bố!)
- **PACF:** Con giống Ông **trực tiếp** (bỏ qua ảnh hưởng của Bố)? → Có thể ít hơn

**PACF loại bỏ ảnh hưởng gián tiếp**, chỉ giữ tương quan trực tiếp.

### 📐 Ý nghĩa:

- **PACF(1):** = ACF(1) (không có gì để loại)
- **PACF(2):** Tương quan Y(t) với Y(t-2), **sau khi loại bỏ** ảnh hưởng của Y(t-1)
- **PACF(3):** Tương quan Y(t) với Y(t-3), **sau khi loại bỏ** ảnh hưởng của Y(t-1) và Y(t-2)

## 4.3 Sử Dụng ACF/PACF Để Xác Định Model

### 🔑 Quy tắc vàng:

| Pattern | ACF | PACF | → Model |
|---------|-----|------|---------|
| **AR(p)** | Giảm dần (decays) | **Cắt ngang** sau lag p | Dùng AR(p) |
| **MA(q)** | **Cắt ngang** sau lag q | Giảm dần (decays) | Dùng MA(q) |
| **ARMA(p,q)** | Giảm dần | Giảm dần | Dùng ARMA(p,q) |

### 🧠 Cách nhớ:
- **PACF cutoff at lag p** → p là order của **AR**
- **ACF cutoff at lag q** → q là order của **MA**
- Cả hai giảm dần → Dùng cả AR + MA

### 📊 Confidence Interval (Vùng xanh):
- Các thanh nằm **trong vùng xanh** → KHÔNG significant
- Các thanh **vượt ra ngoài** vùng xanh → SIGNIFICANT

---
## 🔬 Phần Thực Hành
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf, pacf
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('✅ Import thành công!')

### 📝 Ví dụ 1: ACF của Các Loại Process Khác Nhau

In [ ]:
np.random.seed(42)
n = 500

# 1. White Noise (không có tương quan)
white_noise = np.random.normal(0, 1, n)

# 2. AR(1) process: Y(t) = 0.8 * Y(t-1) + error
ar1 = [0]
for i in range(1, n):
    ar1.append(0.8 * ar1[i-1] + np.random.normal(0, 1))
ar1 = np.array(ar1)

# 3. MA(1) process: Y(t) = error(t) + 0.8 * error(t-1)
errors = np.random.normal(0, 1, n+1)
ma1 = np.array([errors[i] + 0.8 * errors[i-1] for i in range(1, n+1)])

# 4. Random Walk (non-stationary)
random_walk = np.cumsum(np.random.normal(0, 1, n))

# Plot ACF cho từng loại
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

data_dict = {
    'White Noise': white_noise,
    'AR(1): φ=0.8': ar1,
    'MA(1): θ=0.8': ma1,
    'Random Walk': random_walk
}

for idx, (name, data) in enumerate(data_dict.items()):
    # Time series plot
    axes[0, idx].plot(data, linewidth=0.5)
    axes[0, idx].set_title(name, fontweight='bold')
    
    # ACF plot
    plot_acf(data, ax=axes[1, idx], lags=30, alpha=0.05)
    axes[1, idx].set_title(f'ACF - {name}', fontweight='bold')

plt.suptitle('📊 ACF Patterns cho Các Loại Process', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('🔍 Nhận xét:')
print('  White Noise: ACF ≈ 0 ở mọi lag → Không có tương quan')
print('  AR(1):       ACF giảm dần (exponential decay)')
print('  MA(1):       ACF cắt ngang sau lag 1 → q=1')
print('  Random Walk: ACF giảm rất chậm → NON-STATIONARY!')

### 📝 Ví dụ 2: PACF và Cách Đọc

In [ ]:
# Tạo AR(2): Y(t) = 0.6*Y(t-1) + 0.3*Y(t-2) + error
np.random.seed(42)
n = 500

ar2 = [0, 0]
for i in range(2, n):
    ar2.append(0.6 * ar2[i-1] + 0.3 * ar2[i-2] + np.random.normal(0, 1))
ar2 = np.array(ar2)

# Tạo MA(2): Y(t) = error + 0.5*error(t-1) + 0.3*error(t-2)
errors = np.random.normal(0, 1, n+2)
ma2 = np.array([errors[i] + 0.5*errors[i-1] + 0.3*errors[i-2] for i in range(2, n+2)])

# Plot ACF + PACF side by side
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

# AR(2)
axes[0, 0].plot(ar2, linewidth=0.5, color='steelblue')
axes[0, 0].set_title('AR(2) Time Series', fontweight='bold')
plot_acf(ar2, ax=axes[0, 1], lags=20)
axes[0, 1].set_title('ACF - AR(2)\n(Decays gradually)', fontweight='bold', color='blue')
plot_pacf(ar2, ax=axes[0, 2], lags=20)
axes[0, 2].set_title('PACF - AR(2)\n(Cuts off after lag 2! ✅)', fontweight='bold', color='red')
axes[0, 3].text(0.1, 0.5, 'ACF: Decays\nPACF: Cutoff at 2\n\n→ AR(p=2)\n\n📌 PACF tells p!', 
                fontsize=14, verticalalignment='center', 
                bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[0, 3].axis('off')

# MA(2)
axes[1, 0].plot(ma2, linewidth=0.5, color='coral')
axes[1, 0].set_title('MA(2) Time Series', fontweight='bold')
plot_acf(ma2, ax=axes[1, 1], lags=20)
axes[1, 1].set_title('ACF - MA(2)\n(Cuts off after lag 2! ✅)', fontweight='bold', color='red')
plot_pacf(ma2, ax=axes[1, 2], lags=20)
axes[1, 2].set_title('PACF - MA(2)\n(Decays gradually)', fontweight='bold', color='blue')
axes[1, 3].text(0.1, 0.5, 'ACF: Cutoff at 2\nPACF: Decays\n\n→ MA(q=2)\n\n📌 ACF tells q!', 
                fontsize=14, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[1, 3].axis('off')

plt.suptitle('🔑 Quy Tắc Vàng: PACF → p (AR), ACF → q (MA)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 📝 Ví dụ 3: Nhận Diện Seasonality Từ ACF

In [ ]:
# Tạo dữ liệu có seasonal pattern
np.random.seed(42)
n = 365 * 2
t = np.arange(n)

# Weekly seasonality
weekly = 10 * np.sin(2 * np.pi * t / 7) + np.random.normal(0, 2, n)

# Monthly seasonality  
monthly = 20 * np.sin(2 * np.pi * t / 30) + np.random.normal(0, 3, n)

# Yearly seasonality
yearly = 30 * np.sin(2 * np.pi * t / 365) + np.random.normal(0, 5, n)

# Plot ACF to detect seasonality
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for idx, (data, name, period) in enumerate([
    (weekly, 'Weekly Seasonality (period=7)', 50),
    (monthly, 'Monthly Seasonality (period=30)', 100),
    (yearly, 'Yearly Seasonality (period=365)', 400)
]):
    axes[idx, 0].plot(data[:200], linewidth=0.5)
    axes[idx, 0].set_title(f'{name}', fontweight='bold')
    
    plot_acf(data, ax=axes[idx, 1], lags=period)
    axes[idx, 1].set_title(f'ACF - {name}', fontweight='bold')

plt.suptitle('🔍 ACF Phát Hiện Seasonality: Peaks Lặp Lại Tại Các Lag Bội Số Của Period', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 ACF giúp phát hiện seasonality:')
print('   Weekly:  Peaks tại lag 7, 14, 21, 28...')
print('   Monthly: Peaks tại lag 30, 60, 90...')
print('   Yearly:  Peaks tại lag 365, 730...')

### 📝 Ví dụ 4: Hướng Dẫn Chọn Parameters p, d, q

In [ ]:
def suggest_arima_params(series, max_lags=20):
    """
    Phân tích ACF/PACF và đề xuất tham số ARIMA(p,d,q)
    """
    from statsmodels.tsa.stattools import adfuller
    
    print('='*60)
    print('  PHÂN TÍCH ACF/PACF → ĐỀ XUẤT ARIMA(p,d,q)')
    print('='*60)
    
    # Bước 1: Xác định d
    print('\n📌 Bước 1: Xác định d (differencing order)')
    d = 0
    temp = series.copy()
    for i in range(3):
        p_val = adfuller(temp.dropna())[1]
        if p_val < 0.05:
            print(f'   d={i}: ADF p-value = {p_val:.4f} → STATIONARY ✅')
            d = i
            break
        else:
            print(f'   d={i}: ADF p-value = {p_val:.4f} → NON-STATIONARY ❌ → Diff thêm')
            temp = temp.diff().dropna()
            d = i + 1
    
    print(f'   → d = {d}')
    
    # Bước 2: Plot ACF/PACF trên data đã stationary
    stationary_data = series.copy()
    for _ in range(d):
        stationary_data = stationary_data.diff().dropna()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(stationary_data, ax=axes[0], lags=max_lags, alpha=0.05)
    axes[0].set_title(f'ACF (after d={d})', fontweight='bold')
    plot_pacf(stationary_data, ax=axes[1], lags=max_lags, alpha=0.05)
    axes[1].set_title(f'PACF (after d={d})', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Bước 3: Xác định p và q
    acf_vals = acf(stationary_data, nlags=max_lags)
    pacf_vals = pacf(stationary_data, nlags=max_lags)
    
    # Confidence bound
    conf = 1.96 / np.sqrt(len(stationary_data))
    
    # Tìm q từ ACF (lag đầu tiên nằm trong confidence interval)
    q = 0
    for i in range(1, len(acf_vals)):
        if abs(acf_vals[i]) > conf:
            q = i
        else:
            break
    
    # Tìm p từ PACF
    p = 0
    for i in range(1, len(pacf_vals)):
        if abs(pacf_vals[i]) > conf:
            p = i
        else:
            break
    
    print(f'\n📌 Bước 2: Xác định p và q')
    print(f'   Nhìn PACF → p ≈ {p} (significant lags trong PACF)')
    print(f'   Nhìn ACF  → q ≈ {q} (significant lags trong ACF)')
    print(f'\n🎯 ĐỀ XUẤT: ARIMA({p}, {d}, {q})')
    print(f'   Hoặc thử: ARIMA({max(p-1,0)}, {d}, {max(q-1,0)}) hoặc ARIMA({p+1}, {d}, {q+1})')
    
    return p, d, q

# Test
np.random.seed(42)
# Tạo ARIMA(2,1,1) process
y = np.cumsum(np.random.normal(0, 1, 300))  # Random walk
test_series = pd.Series(y)

p, d, q = suggest_arima_params(test_series)

### 📝 Ví dụ 5: ACF/PACF Trên Dữ Liệu Thực

In [ ]:
# Đọc dữ liệu thực
data_retail = pd.read_csv('../Bai thi thu/Course Files/retail_sales_dataset.csv')

# Xử lý
date_cols = [col for col in data_retail.columns if 'date' in col.lower()]
print(f'Các cột: {data_retail.columns.tolist()}')
print(f'Cột date: {date_cols}')
print(f'\n5 dòng đầu:')
data_retail.head()

In [ ]:
# Tìm cột số để phân tích
numeric_cols = data_retail.select_dtypes(include=[np.number]).columns.tolist()
print(f'Các cột số: {numeric_cols}')

# Chọn cột đầu tiên có giá trị phù hợp
if len(numeric_cols) > 0:
    target_col = numeric_cols[0]
    print(f'\nPhân tích cột: {target_col}')
    
    series = data_retail[target_col].dropna()
    
    # Plot ACF/PACF
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Original
    axes[0, 0].plot(series.values, linewidth=0.5)
    axes[0, 0].set_title(f'Original: {target_col}', fontweight='bold')
    
    plot_acf(series, ax=axes[0, 1], lags=40)
    axes[0, 1].set_title('ACF - Original', fontweight='bold')
    
    # After differencing
    diff_series = series.diff().dropna()
    axes[1, 0].plot(diff_series.values, linewidth=0.5)
    axes[1, 0].set_title(f'After Differencing (d=1)', fontweight='bold')
    
    plot_pacf(diff_series, ax=axes[1, 1], lags=40)
    axes[1, 1].set_title('PACF - After Diff', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

---
## 🏋️ BÀI TẬP THỰC HÀNH
---

### Bài 1: Nhận Diện Process (⭐ Dễ)

Tạo các process sau và vẽ ACF/PACF:
1. AR(1) với φ = 0.9
2. AR(1) với φ = -0.5
3. MA(1) với θ = 0.7
4. MA(2) với θ₁ = 0.5, θ₂ = -0.3

Nhận xét sự khác biệt ACF/PACF giữa các process.

In [ ]:
# TODO: Viết code ở đây

### Bài 2: Phát Hiện Seasonality (⭐⭐ Trung bình)

Tạo dữ liệu có **cả weekly VÀ monthly seasonality**:
```python
y = 10*sin(2π*t/7) + 20*sin(2π*t/30) + noise
```

1. Vẽ ACF với lags=100
2. Xác định các seasonal period từ ACF
3. Nhận xét: Có thể phát hiện multiple seasonality từ ACF không?

In [ ]:
# TODO: Viết code ở đây

### Bài 3: Xác Định ARIMA Parameters (⭐⭐⭐ Nâng cao)

Với file `stores_sales_forecasting.csv`:
1. Đọc và xử lý dữ liệu
2. Kiểm tra stationarity → Xác định d
3. Vẽ ACF/PACF trên data đã stationary
4. Đề xuất p, d, q cho ARIMA model
5. So sánh đề xuất của bạn với `auto_arima` (nếu có pmdarima)

In [ ]:
# TODO: Viết code ở đây